[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ushaloy/SAVE-LEWS/blob/main/notebooks/06_robustness_ablation_alerts_nowcast.ipynb)

# SAVE-LEWS — post hoc analyses: physics-guidance ablation, robustness, alerts and nowcast degradation

These analyses were run **after** the main (and, in Puerto Rico, pre-registered) analysis and are labelled post hoc in the manuscript (see `puerto_rico/DEVIATIONS.md`).

| Analysis | Script | Stored output | Re-run time (CPU) |
|---|---|---|---|
| Thai ablation U / bounds-only / PG-MLP | `thailand/pg_ablation.py U|B|PG`, `summarize_ablation.py` | `thailand/res_ablation_*.json`, `ablation_summary.json` | ~20 min per variant |
| Puerto Rico unconstrained MLP | `puerto_rico/run_p1_mlpU.py <site>`, `eval_mlpU.py` | `out/p1_mlpU_*.npz`, `out/mlpU_summary.json` | ~15 min per site |
| Se-normalisation robustness, nested selection, Holm | `se_robust.py`, `se_nested.py <site>`, `se_nested2.py` | `out/se_*.json` | ~1–2 h per site |
| Thai alerts | `thai_alert_preds.py`, `thai_alerts.py` | `thailand/thai_alerts.json` | ~5 min |
| Puerto Rico alerts | `pr_alerts_rise.py` | `out/pr_alerts_rise.json` | ~1 min |
| Nowcast degradation | `thailand/nowcast_degrade.py` | `thailand/nowcast_degrade.json` | ~20 min |

By default the notebook loads the stored outputs. Set the `RERUN_*` flags to recompute.

In [1]:
import os, sys, subprocess
REPO_URL = 'https://github.com/Ushaloy/SAVE-LEWS'   # <-- set this to your GitHub repository before running in Colab
def _repo_root():
    here = os.getcwd()
    for c in [here, os.path.dirname(here), '/content/SAVE-LEWS', os.path.join(here, 'SAVE-LEWS')]:
        if os.path.exists(os.path.join(c, 'thailand', 'data_pipeline.py')):
            return c
    dst = '/content/SAVE-LEWS' if 'google.colab' in sys.modules else os.path.join(here, 'SAVE-LEWS')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, dst], check=True)
    return dst
ROOT = _repo_root(); os.chdir(os.path.join(ROOT, '.'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', os.path.join(ROOT, 'requirements.txt')], check=True)
print('repository:', ROOT)
RERUN_ABLATION = False   # Thai + Puerto Rico ablation (~1.5 h)
RERUN_ROBUST = False     # Se robustness and nested selection (~3 h)
RERUN_ALERTS = False     # alert scoring (~10 min)
RERUN_NOWCAST = False    # nowcast degradation (~20 min)
import json, pandas as pd
pd.set_option('display.width', 220)
def sh(script, *args, cwd='.'):
    print('>>', cwd, script, *args); subprocess.run([sys.executable, script, *args], cwd=cwd, check=True)

repository: /content/SAVE-LEWS


## 1. Physics-guidance ladder (Table 6, Fig. 7)

In [2]:
if RERUN_ABLATION:
    for v in ['U', 'B', 'PG']: sh('pg_ablation.py', v, cwd='thailand')
    for s in ['utuado', 'toronegro']: sh('run_p1_mlpU.py', s, cwd='puerto_rico')
sh('summarize_ablation.py', cwd='thailand'); sh('eval_mlpU.py', cwd='puerto_rico')
A = json.load(open('thailand/ablation_summary.json'))
display(pd.DataFrame({k: {c: v[c] for c in ['oracle60', 'past60', 'oracle_viol_monotone', 'oracle_viol_wet_dry']} for k, v in A.items()}).T)
M = json.load(open('puerto_rico/out/mlpU_summary.json'))
display(pd.DataFrame({k: v['skill'] for k, v in M.items()}).round(3))

>> thailand summarize_ablation.py
{
 "U": {
  "oracle30": [
   0.09,
   0.32
  ],
  "oracle60": [
   0.17,
   0.37
  ],
  "oracle120": [
   0.53,
   0.64
  ],
  "oracle_s2_60": [
   -33.24,
   -25.26,
   -9.59
  ],
  "zs_oracle": [
   [
    0.87,
    0.66,
    0.81
   ],
   [
    0.35,
    0.19,
    0.23
   ],
   [
    -1.88,
    -0.05,
    -1.31
   ],
   [
    0.69,
    0.67,
    0.65
   ]
  ],
  "oracle_viol_bounds": 0.0068,
  "oracle_viol_wet_dry": 0.1457,
  "oracle_viol_monotone": 0.422,
  "past30": [
   -1.08,
   -0.24
  ],
  "past60": [
   -0.77,
   -0.2
  ],
  "past120": [
   -0.83,
   -0.31
  ],
  "past_s2_60": [
   -15.68,
   -47.07,
   -31.78
  ],
  "zs_past": [
   [
    0.11,
    0.04,
    0.23
   ],
   [
    -0.01,
    -0.03,
    -0.08
   ],
   [
    -0.85,
    -15.54,
    -4.54
   ],
   [
    0.34,
    0.19,
    0.25
   ]
  ],
  "past_viol_bounds": 0.0045,
  "past_viol_wet_dry": 0.3535,
  "past_viol_monotone": null
 },
 "B": {
  "oracle30": [
   0.23,
   0.33
  ],
  "oracl

{
 "utuado primary": {
  "n_storms": 89,
  "skill": {
   "R-cal": 0.004,
   "R-lab": -1.268,
   "R-cal-Se (exploratory)": 0.17,
   "PG-MLP s0": 0.389,
   "PG-MLP s1": 0.313,
   "PG-MLP s2": 0.34,
   "U s0": 0.103,
   "U s1": 0.211,
   "U s2": 0.075
  }
 },
 "utuado secondary": {
  "n_storms": 89,
  "skill": {
   "R-cal": 0.492,
   "R-lab": 0.247,
   "R-cal-Se (exploratory)": 0.665,
   "PG-MLP s0": 0.485,
   "PG-MLP s1": 0.448,
   "PG-MLP s2": 0.461,
   "U s0": 0.481,
   "U s1": 0.433,
   "U s2": 0.464
  }
 },
 "toronegro primary": {
  "n_storms": 51,
  "skill": {
   "R-cal": -1.159,
   "R-lab": -7.682,
   "R-cal-Se (exploratory)": -0.256,
   "PG-MLP s0": 0.73,
   "PG-MLP s1": 0.708,
   "PG-MLP s2": 0.715,
   "U s0": 0.558,
   "U s1": 0.551,
   "U s2": 0.577
  }
 },
 "toronegro secondary": {
  "n_storms": 51,
  "skill": {
   "R-cal": -0.282,
   "R-lab": -15.448,
   "R-cal-Se (exploratory)": -0.252,
   "PG-MLP s0": 0.615,
   "PG-MLP s1": 0.613,
   "PG-MLP s2": 0.605,
   "U s0": 0.481,
  

,oracle60,past60,oracle_viol_monotone,oracle_viol_wet_dry
U,"[0.17, 0.37]","[-0.77, -0.2]",0.422,0.1457
B,"[0.32, 0.42]","[0.1, 0.23]",0.0004,0.1096
PG,"[0.46, 0.54]","[-0.25, 0.13]",0.0002,0.0283


,utuado primary,utuado secondary,toronegro primary,toronegro secondary
R-cal,0.004,0.492,-1.159,-0.282
R-lab,-1.268,0.247,-7.682,-15.448
R-cal-Se (exploratory),0.170,0.665,-0.256,-0.252
PG-MLP s0,0.389,0.485,0.730,0.615
PG-MLP s1,0.313,0.448,0.708,0.613
PG-MLP s2,0.340,0.461,0.715,0.605
U s0,0.103,0.481,0.558,0.481
U s1,0.211,0.433,0.551,0.433
U s2,0.075,0.464,0.577,0.442


## 2. Robustness of the exploratory R-cal-Se result at Utuado (Tables 8 and S7)

In [3]:
if RERUN_ROBUST:
    sh('se_robust.py', cwd='puerto_rico')
    for s in ['utuado', 'toronegro']: sh('se_nested.py', s, cwd='puerto_rico')
    sh('se_nested2.py', cwd='puerto_rico')
R = json.load(open('puerto_rico/out/se_robust.json'))
display(pd.DataFrame({w: d['skill'] for w, d in R.items()}).round(3))
for s in ['utuado', 'toronegro']:
    N = json.load(open(f'puerto_rico/out/se_nested_{s}.json'))
    print(s, 'nested-selected ranges:', sorted(set(map(tuple, N['chosen']))),
          '| skill:', {w: round(N[w]['skill']['Se nested'], 2) for w in ['secondary', 'primary']})
print('Holm-corrected one-sided p:', json.load(open('puerto_rico/out/se_holm.json')))

,secondary,primary
R-cal,0.492,0.004
R-lab,0.247,-1.268
R-cal-Se (exploratory),0.665,0.170
PG-MLP s0,0.485,0.389
PG-MLP s1,0.448,0.313
PG-MLP s2,0.461,0.340
generic fixed (0.5–99.5%),0.548,0.138
lab 22 cm fixed (0.5–99.5%),0.287,0.087
lab 57 cm fixed (0.5–99.5%),0.551,0.067
lab 92 cm fixed (0.5–99.5%),0.121,0.069


utuado nested-selected ranges: [(0.05, 0.95)] | skill: {'secondary': 0.71, 'primary': 0.68}
toronegro nested-selected ranges: [(0.005, 0.995), (0.01, 0.99)] | skill: {'secondary': -1.0, 'primary': -0.71}
Holm-corrected one-sided p: {'utuado secondary': 0.0004, 'utuado primary': 0.9895, 'toronegro secondary': 1.0, 'toronegro primary': 1.0}


## 3. Hydrological alert states (Level 2; Table 9, Fig. 10) — not slope-failure warnings

In [4]:
if RERUN_ALERTS:
    sh('thai_alert_preds.py', cwd='thailand'); sh('thai_alerts.py', cwd='thailand'); sh('pr_alerts_rise.py', cwd='puerto_rico')
T = json.load(open('thailand/thai_alerts.json'))['0.01']
display(pd.DataFrame(T).T.round(2))
P = json.load(open('puerto_rico/out/pr_alerts_rise.json'))
display(pd.DataFrame({(s, t, m): r for s, d in P.items() for t, v in d.items() for m, r in v['results'].items()}).T[['POD', 'FAR', 'CSI', 'POD_timely', 'median_timing_h']].round(2))

,events,N,POD,FAR,CSI,EI,MAR
PG-MLP,118.0,907.0,0.85,0.28,0.64,0.94,0.15
M0,118.0,907.0,0.06,0.00,0.06,0.88,0.94
M2,118.0,907.0,0.75,0.31,0.56,0.92,0.25
M2np,118.0,907.0,0.79,0.19,0.66,0.95,0.21
M3,118.0,907.0,0.18,0.19,0.17,0.89,0.82
B-TS,118.0,907.0,0.38,0.18,0.35,0.91,0.62


POD   FAR   CSI  POD_timely  median_timing_h
utuado    57 cm R-cal                   0.52  0.15  0.48        0.02            20.00
                R-cal-Se (exploratory)  0.79  0.06  0.75        0.46             0.50
                PG-MLP s0               0.84  0.13  0.75        0.61             0.25
                PG-MLP s1               0.82  0.15  0.72        0.61             0.25
                PG-MLP s2               0.84  0.13  0.75        0.61             0.25
          72 cm R-cal                   0.89  0.31  0.64        0.02            13.12
                R-cal-Se (exploratory)  0.88  0.25  0.68        0.36            -0.25
                PG-MLP s0               0.96  0.17  0.81        0.61             0.00
                PG-MLP s1               0.95  0.17  0.79        0.54             0.00
                PG-MLP s2               0.93  0.16  0.79        0.59            -0.25
toronegro 90 cm R-cal                   1.00  0.10  0.90        0.12           -25.00
                R-cal-Se (exploratory)  0.56  0.04  0.55        0.35             0.12
                PG-MLP s0               0.86  0.14  0.76        0.77             0.00
                PG-MLP s1               0.86  0.14  0.76        0.70             0.00
                PG-MLP s2               0.86  0.14  0.76        0.74             0.00

## 4. Sensitivity to the rain nowcast (Results, Step III)

In [5]:
if RERUN_NOWCAST:
    sh('nowcast_degrade.py', cwd='thailand')
pd.Series(json.load(open('thailand/nowcast_degrade.json'))).round(2).to_frame('PG-MLP 60-min skill')

,PG-MLP 60-min skill
oracle,0.54
"amount error, factor ~1.6 (sigma 0.5)",0.46
"amount error, factor ~2.7 (sigma 1.0)",0.26
timing error +30 min,0.33
timing error +60 min,0.12
missed 30% of rain,0.52
past-only model,0.02
